# Lesson 13 - Camera Colour Ball Tracking + Push Into Goal V2

Goal:
- use the V2 robot object
- take camera pictures in the notebook
- calibrate one target colour
- detect where the ball is in the image
- choose a movement from the camera result

The robot should sense first, then decide how to move.

In [ ]:
from lesson_header import *
show_v2_status()
myRobot = bot(base_speed=300)

## Take One Camera Picture
Start with a still frame. Do not move the robot until you know the camera can see the ball.

In [ ]:
from lesson_header import *
myRobot = bot(base_speed=300)

picture = myRobot.vision.capture()
frame = picture['frame_bgr']
print('frame size:', frame.shape)

## Aim The Camera Head
Move only the camera head while you search. Centre the camera again before calibrating colour.

In [ ]:
from lesson_header import *
myRobot = bot(base_speed=300)

myRobot.camera.center_all()
# Try one of these if the ball is off to the side.
# myRobot.camera.glance_left(amplitude=180, hold_s=0.2)
# myRobot.camera.glance_right(amplitude=180, hold_s=0.2)
# myRobot.camera.look_down(amplitude=120, hold_s=0.2)

myRobot.vision.capture()

## Calibrate The Ball Colour
Put the target ball in the middle of the camera image, then calibrate that colour.

Use a colour name that matches your target ball, such as `green`, `red`, `blue`, or `pink`.

In [ ]:
from lesson_header import *
myRobot = bot(base_speed=300)

TARGET_COLOUR = 'green'
myRobot.vision.calibrate_color(TARGET_COLOUR)
myRobot.vision.show_color(TARGET_COLOUR)
myRobot.vision.show_profiles()

## Read The Ball Position
`show_color` returns a list of detected objects. Use the centre `cx` value to decide whether the ball is left, right, or centred.

In [ ]:
from lesson_header import *
myRobot = bot(base_speed=300)

result = myRobot.vision.show_color(TARGET_COLOUR)
objects = result['objects']

if not objects:
    print('No ball found. Move closer, aim the camera, or recalibrate.')
else:
    ball = max(objects, key=lambda item: item['area'])
    frame_width = 640
    mid_x = frame_width // 2
    error = ball['cx'] - mid_x
    print('largest ball:', ball)
    print('x error from centre:', error)

## Decide One Movement
This is a single safe decision step. Test it before making a loop.

In [ ]:
from lesson_header import *
myRobot = bot(base_speed=300)

DEADZONE_PX = 45

result = myRobot.vision.show_color(TARGET_COLOUR)
objects = result['objects']

if not objects:
    print('Decision: stop, because no target was found')
    myRobot.move.stop()
else:
    ball = max(objects, key=lambda item: item['area'])
    mid_x = 640 // 2
    error = ball['cx'] - mid_x

    if error < -DEADZONE_PX:
        print('Decision: move left')
        myRobot.move.left(seconds=0.12, speed=80)
    elif error > DEADZONE_PX:
        print('Decision: move right')
        myRobot.move.right(seconds=0.12, speed=80)
    else:
        print('Decision: push forward')
        myRobot.move.forward(seconds=0.18, speed=90)

    myRobot.move.stop()

## Build A Short Tracking Loop
Run a few sense-decide-move cycles. Keep the loop short while testing.

In [ ]:
from lesson_header import *
import time

myRobot = bot(base_speed=300)

for step in range(6):
    result = myRobot.vision.show_color(TARGET_COLOUR, show=False)
    objects = result['objects']

    if not objects:
        print(step, 'lost target')
        myRobot.move.stop()
        time.sleep(0.2)
        continue

    ball = max(objects, key=lambda item: item['area'])
    error = ball['cx'] - 320
    print(step, 'cx=', ball['cx'], 'area=', int(ball['area']), 'error=', error)

    if error < -DEADZONE_PX:
        myRobot.move.left(seconds=0.08, speed=75)
    elif error > DEADZONE_PX:
        myRobot.move.right(seconds=0.08, speed=75)
    else:
        myRobot.move.forward(seconds=0.12, speed=85)

    time.sleep(0.1)

myRobot.move.stop()
myRobot.vision.show_color(TARGET_COLOUR)

## Optional: Use The Robot Tracking Service
The camera colour examples above are pure notebook logic. The robot also has a tracking service namespace for more advanced experiments.

In [ ]:
from lesson_header import *
myRobot = bot(base_speed=300)

# These calls depend on the tracking service running on the robot image.
# myRobot.tracking.set_color_name(TARGET_COLOUR)
# myRobot.tracking.start()
# myRobot.tracking.set_pan_tilt(True)
# myRobot.tracking.stop()

## What To Improve
After the first working version, think about:
- how to stop overshooting
- what to do when the ball is lost
- when to search with the camera head instead of moving the whole robot
- how to make the push stage more reliable